### [ 모델 완성 후 웹/앱 서비스 진행 시 ]

- 필요한 것
    * 모델 파일( .pt, .pth )
    * 전처리기 즉, Encoder, Scaler, pipeline .... 저장한 파일(.pkl)
    * 데이터의 컬럼명, 타겟 저장한 파일(.pkl)
    * 모델 및 데이터셋의 클래스 선언 파일 (py)

In [1]:
## ----------------------------------------------------
## 모듈 로딩
## ----------------------------------------------------
import joblib                           ## 모델 외 부가 파일들 로딩 
import pandas as pd                     ## 새로운 데이터 생성 시 
import numpy as np 
import torch                            ## 모델 로딩 및 텐서용

import sys
sys.path.append(r'D:\KDT\VS_KDT_14\[9]_DL\DAY05')
from dnn_fish_class import WeightRegression
from reg_func import predict_regression

In [2]:
## ----------------------------------------------------
## 로딩할 모델, 전처리기, 데이터 관련 설정
## ----------------------------------------------------
## => 저장 폴더
MODEL_DIR   = '../Models'

## => 전처리기 : 스케일러 2개
F_SCALER    = 'featureScaler.pkl'           ## 입력 피쳐용 스케일러
T_SCALER    = 'targetScaler.pkl'            ## 타겟용 스케일러

## => 부가데이터 
F_COLS      =  'featureColos.pkl'           ## 입력 피쳐의 컬럼명 

## => 모델 파일
PARAMS_FILE = 'fish_model_weights.pth'      ## 층별 파라미터(W, b) 만 저장 
ALL_FILE    = 'fish_model_all.pt'           ## 모델 구조 + 층별 파라미터(W, b) 모두 저장


In [3]:
## ----------------------------------------------------
## 인스턴스로 로딩
## ----------------------------------------------------
## => [1] 모델: torch.load(경로+파일명)
model = torch.load(f'{MODEL_DIR}/{ALL_FILE}', weights_only=False) 

## => [2] 전처리기 로딩 : joblib.load(경로+파일명)
f_scaler = joblib.load(f'{MODEL_DIR}/{F_SCALER}')
t_scaler = joblib.load(f'{MODEL_DIR}/{T_SCALER}')

## => [3] 부가 데이터 로딩 : joblib.load(경로+파일명)
f_columns = joblib.load(f'{MODEL_DIR}/{F_COLS}')

In [4]:
print("[모델 구조]=====", model, sep='\n')

[모델 구조]=====
WeightRegression(
  (hd1_layer): Linear(in_features=4, out_features=3, bias=True)
  (out_layer): Linear(in_features=3, out_features=1, bias=True)
)


In [5]:
print("[스케일러]=====", f_scaler.mean_, f_scaler.scale_, f_scaler.feature_names_in_, sep='\n')
print("[스케일러]=====", t_scaler.mean_, t_scaler.scale_, t_scaler.feature_names_in_, sep='\n')


[스케일러]=====
[29.776    32.675     9.301853  4.607534]
[10.46930867 11.27569399  4.18369442  1.65377659]
['Length' 'Diagonal' 'Height' 'Width']
[스케일러]=====
[434.869]
[359.92531245]
['Weight']


In [6]:
print("[부가데이터]=====", f_columns, sep='\n')

[부가데이터]=====
['Length', 'Diagonal', 'Height', 'Width']


>> **새로운 데이터/사용자 입력 데이터 예측 서비스**

In [7]:
## 정답 Weight = 242인 데이터라고 가정
## 피쳐 : Length, Diagonal, Height, Width
new_data = pd.DataFrame( [[25.4, 30, 11.52, 4.02]], columns=f_columns )

## 학습용 데이터셋기반 피쳐 스케일러 사용 
scaled_data = f_scaler.transform(new_data)

## Tensor 변환
data = torch.tensor(scaled_data, dtype=torch.float32)

In [8]:
## 예측 : 타겟 스케일러로 복원된 예측값 반환
device      = 'cuda' if torch.cuda.is_available() else 'cpu'
pred_weight = predict_regression( data, model, t_scaler, device)

print(f'예측 무게 : {pred_weight[0][0]}')

예측 무게 : 431.598876953125
